# Permuting and sampling data to calculate the null hypothesis

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import warnings; warnings.filterwarnings('ignore')

In [5]:
data_loc = '../data/server_ready'

files = [os.path.join(data_loc, f) for f in os.listdir(data_loc) if (not f.startswith('._')) and f.endswith('.tsv')]
print(files)

['../data/server_ready/cha_corpus-callfriend.tsv', '../data/server_ready/cha_corpus-callhome.tsv', '../data/server_ready/callfriend-ko_corpus.tsv', '../data/server_ready/cha_corpus-yiddish.tsv', '../data/server_ready/d_cha_corpus-croatian.tsv', '../data/server_ready/finchat_corpus.tsv']


In [6]:
null_output_file = 'null-corpora/{}-null-corpus.tsv'
null_output_file = os.path.join(data_loc, null_output_file)
null_output_file

'../data/server_ready/null-corpora/{}-null-corpus.tsv'

## Setting up the null-test data

### Load, permute, save

In [7]:
for f in tqdm(files):
    print(f)
    # df = pd.read_csv(f)
    df = pd.read_table(f, sep='\t')

    # print(df[['x_utterance', 'y_utterance']].isna().sum(),'\n')
    df = df.loc[
        (~df['x_utterance'].isna())
        & (~df['y_utterance'].isna())
    ]
    # print(df[['x_utterance', 'y_utterance']].isna().sum())

    print('permuting sentences')

    df['y_utterance'] = df['y_utterance'].sample(frac=1.0).to_list()
    # print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    df['x_turn_id'] = df['conversation_id'].astype(str) + '-' + df['x_turn_id'].astype(str)

    df['sample_delta'] = df.groupby(by=['x_turn_id', 'self']).cumcount() + 1


    print('finding examples')
    # a lot of work just to find useful examples ...
    good_examples = df[['corpus', 'x_turn_id', 'self', 'sample_delta']].groupby(by=['corpus', 'x_turn_id', 'self', 'sample_delta']).agg('max').reset_index()
    good_examples = pd.merge(
        left=good_examples.loc[good_examples['self']], left_on='x_turn_id',
        right=good_examples[['x_turn_id', 'self', 'sample_delta']].loc[~good_examples['self']], right_on='x_turn_id',
        how='left'
    )
    good_examples = good_examples.loc[
        ((good_examples['sample_delta_x'] >= 5)
        & (good_examples['sample_delta_x'] <= 70))
        & ((good_examples['sample_delta_y'] >= 5)
        & (good_examples['sample_delta_y'] <= 70))
    ]

    df = df.loc[df['x_turn_id'].isin(good_examples['x_turn_id'])]
    # print(df[['x_utterance', 'y_utterance']].isna().sum(), '\n')

    print('saving data')
    for c in tqdm(df['corpus'].unique()):

        sub = df['x_turn_id'].loc[df['corpus'].isin([c])].unique()

        k = int(np.ceil(len(sub)*.1))

        if k > 0:
            sub = np.random.choice(sub, size=(k,), replace=False)
            df_ = df.loc[df['x_turn_id'].isin(sub)]

            print('\n', c, k, df_.shape, '\n', df_[['x_utterance', 'y_utterance']].isna().sum())

            null_output_file_ = str(null_output_file).format(c)
            # save subset to null output file.
            # if os.path.exists(null_output_file):
            #     df_.to_csv(null_output_file, index=False, header=False, encoding='utf-8', mode='a', sep='\t')
            # else:
            df_.to_csv(null_output_file_, index=False, encoding='utf-8', sep='\t')
    print('===][===')

  0%|          | 0/6 [00:00<?, ?it/s]

../data/server_ready/cha_corpus-callfriend.tsv
permuting sentences
finding examples
saving data


  0%|          | 0/4 [00:00<?, ?it/s]


 callfriend-eng-n 1776 (105518, 17) 
 x_utterance    0
y_utterance    0
dtype: int64

 callfriend-eng-s 509 (30086, 17) 
 x_utterance    0
y_utterance    0
dtype: int64

 callfriend-fra-q 3573 (213397, 17) 
 x_utterance    0
y_utterance    0
dtype: int64

 callfriend-spa-s 9248 (556156, 17) 
 x_utterance    0
y_utterance    0
dtype: int64
===][===
../data/server_ready/cha_corpus-callhome.tsv
permuting sentences
finding examples
saving data


  0%|          | 0/5 [00:00<?, ?it/s]

...


 yiddish-yid-men 4 (144, 16) 
 x_utterance    0
y_utterance    0
dtype: int64
===][===
../data/server_ready/d_cha_corpus-croatian.tsv
permuting sentences
finding examples
saving data


  0%|          | 0/1 [00:00<?, ?it/s]


 croation-cro 5675 (325655, 21) 
 x_utterance    0
y_utterance    0
dtype: int64
===][===
../data/server_ready/finchat_corpus.tsv
permuting sentences
finding examples
saving data


  0%|          | 0/1 [00:00<?, ?it/s]


 finchat-fin 263 (9242, 13) 
 x_utterance    0
y_utterance    0
dtype: int64
===][===


### test that file works

In [8]:
corpora_are_in = os.path.join(data_loc, 'null-corpora')
corpora_are_in

'../data/server_ready/null-corpora'

In [9]:
corpora = [f for f in os.listdir(corpora_are_in) if (not f.startswith('._')) and f.endswith('.tsv')]
corpora

['callfriend-eng-n-null-corpus.tsv',
 'callfriend-eng-s-null-corpus.tsv',
 'callfriend-fra-q-null-corpus.tsv',
 'callfriend-spa-s-null-corpus.tsv',
 'callhome-deu-null-corpus.tsv',
 'callhome-eng-null-corpus.tsv',
 'callhome-jpn-null-corpus.tsv',
 'callhome-spa-null-corpus.tsv',
 'callhome-zho-null-corpus.tsv',
 'callfriend-ko-null-corpus.tsv',
 'yiddish-yid-men-null-corpus.tsv',
 'croation-cro-null-corpus.tsv',
 'finchat-fin-null-corpus.tsv']

In [10]:
df = []

In [11]:
for f in corpora:
    df += [pd.read_table(os.path.join(corpora_are_in,f), sep='\t')]
    # print(f, df[-1].shape)
    # print(df[-1][['x_utterance', 'y_utterance']].isna().sum())
    # print('===][===\n')
df = pd.concat(df, ignore_index=True)

In [12]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [14]:
null_output_file = os.path.join(data_loc,'null-corpus.tsv')

In [15]:
df.to_csv(null_output_file, index=False, encoding='utf-8', sep='\t')

#### Test that stitched doc is okay

In [16]:
null_output_file = os.path.join(data_loc,'null-corpus.tsv')

In [18]:
# df = pd.read_csv(null_output_file)
df = pd.read_table(null_output_file, sep='\t')
df.shape

(2555354, 27)

In [19]:
df[['x_utterance', 'y_utterance']].isna().sum()

x_utterance    0
y_utterance    0
dtype: int64

In [20]:
df.head()

,document_line_no,x_turn_id,x_speaker,x_utterance,bullet,recipient,overlapping_text,corpus,conversation_id,com,...,mor,gra,exp,time,err,sit,add,act,wor,TIME
0,64.0,callfriend-eng-n-4175-31,callfriend-eng-n-callfriend-eng-n-4175-M2,hhh hhh hhh well the n s a guy that used to li...,"[53461, 58157]",ADULT,False,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,64.0,callfriend-eng-n-4175-31,callfriend-eng-n-callfriend-eng-n-4175-M2,hhh hhh hhh well the n s a guy that used to li...,"[53461, 58157]",ADULT,False,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,64.0,callfriend-eng-n-4175-31,callfriend-eng-n-callfriend-eng-n-4175-M2,hhh hhh hhh well the n s a guy that used to li...,"[53461, 58157]",ADULT,False,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,64.0,callfriend-eng-n-4175-31,callfriend-eng-n-callfriend-eng-n-4175-M2,hhh hhh hhh well the n s a guy that used to li...,"[53461, 58157]",ADULT,False,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,64.0,callfriend-eng-n-4175-31,callfriend-eng-n-callfriend-eng-n-4175-M2,hhh hhh hhh well the n s a guy that used to li...,"[53461, 58157]",ADULT,False,callfriend-eng-n,callfriend-eng-n-4175,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
